# J25 CWCF2 — validation-only gate audit

Membandingkan checkpoint quarantine yang sama dengan composition gate aktif dan nol. Tidak ada training dan test tetap tertutup.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import importlib, json, os, shutil, subprocess, sys
from pathlib import Path
BRANCH='codex/j25-chromatic-wavelet-composition'
REPO=Path('/content/coffee-bean-detection'); WORK=Path('/content')
os.chdir(WORK)
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96','gdown'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
import torch
if not torch.cuda.is_available(): raise RuntimeError('Aktifkan GPU Colab')
roots=[Path('/content/drive/MyDrive/Coffee_Bean_Detection/experiments'),*Path('/content/drive/.shortcut-targets-by-id').glob('*/Coffee_Bean_Detection/experiments')]
def find_file(folder,relative):
    matches=[root/folder/relative for root in roots if (root/folder/relative).is_file()]
    if not matches: raise FileNotFoundError(f'{folder}/{relative} tidak ditemukan')
    return matches[0]
CWCF1_RESULT=find_file('coffee-standard-j25-cwcf-v1',Path('val_reports/CWCF1_seed42_result.json'))
CWCF2=find_file('coffee-standard-j25-cwcf2-v1',Path('CWCF2/CWCF2_seed42/weights/best.pt'))
QUARANTINE=find_file('coffee-standard-j25-cwcf2-v1',Path('quarantine_reports/CWCF2_seed42_quarantined_result.json'))
OUT=CWCF2.parents[3]; PROJECT=OUT.parents[1]
print('CWCF2:',CWCF2); print('QUARANTINE:',QUARANTINE); print('CWCF1:',CWCF1_RESULT)

In [ ]:
from coffee_detector.analysis.coffee_standard_j25_thesis_provenance import audit_j25_thesis_provenance
from coffee_detector.data.prepare_coffee_standard_j25_source_split import prepare_j25_source_split
ARCHIVE=WORK/'data_aug_11.zip'
if not ARCHIVE.is_file(): subprocess.run([sys.executable,'-m','gdown','https://drive.google.com/uc?id=1AofT7VbiNFM8ul-0vyCAKj7Rp4j5OX0f','-O',str(ARCHIVE)],check=True)
PROVENANCE=WORK/'coffee_standard_j25_thesis_provenance.json'
provenance=audit_j25_thesis_provenance(ARCHIVE,PROVENANCE)
if not provenance['decision'].startswith('PASS'): raise RuntimeError(f'Provenance gagal: {provenance["decision"]}')
DATA=WORK/'coffee-standard-j25-train-siblings-v2'
if DATA.exists(): shutil.rmtree(DATA)
contract=prepare_j25_source_split(ARCHIVE,DATA,seed=42,retain_train_siblings=True)
CONTRACT=DATA/'coffee_standard_j25_train_siblings_summary.json'
print('DATA:',contract['images'],'| GPU:',torch.cuda.get_device_name(0))

In [ ]:
OUTPUT=OUT/'analysis/cwcf2_gate_ablation.json'; LOG=OUT/'analysis/cwcf2_gate_ablation.log'
OUTPUT.parent.mkdir(parents=True,exist_ok=True)
command=[sys.executable,'-u','-m','coffee_detector.analysis.coffee_standard_j25_cwcf2_gate_audit','--data-root',str(DATA),'--development-contract',str(CONTRACT),'--provenance-summary',str(PROVENANCE),'--cwcf2-checkpoint',str(CWCF2),'--quarantine-result',str(QUARANTINE),'--cwcf1-result',str(CWCF1_RESULT),'--output',str(OUTPUT),'--device','0','--authorize-diagnostic']
print('MENJALANKAN VALIDATION-ONLY GATE AUDIT',flush=True)
with LOG.open('w',encoding='utf-8') as stream: process=subprocess.run(command,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT)
if process.returncode:
    print('\n'.join(LOG.read_text(errors='replace').splitlines()[-150:])); raise RuntimeError(f'Audit gagal: {process.returncode}')
result=json.loads(OUTPUT.read_text())
print('GATES:',result['composition_gates'])
print('VALUES:',result['values'])
print('ZERO-ACTIVE:',result['zero_minus_active'])
print('TARGET:',result['target_values'])
print('ATTRIBUTION:',result['attribution'])
print('NEXT:',result['next'],'| TRAINING:',result['training_executed'],'| TEST:',result['test_opened'])